# Module 50: Sparse Tensors

Store only nonzeros with COO/CSR layouts, convert to dense, and run
`torch.sparse.mm` for SpMM-style products.

**Tip:** Always `coalesce()` COO tensors before inspecting unique indices.

In [ ]:
import torch

indices = torch.tensor([[0, 1, 0], [2, 0, 2]])
values = torch.tensor([3.0, 4.0, 1.0])  # (0,2) duplicated
s = torch.sparse_coo_tensor(indices, values, (2, 3))
print("coalesced?", s.is_coalesced())
s = s.coalesce()
print(s.to_dense())
print("values after coalesce:", s.values().tolist())

In [ ]:
dense = torch.tensor([[0.0, 0.0, 3.0], [4.0, 0.0, 5.0]])
csr = dense.to_sparse_csr()
print("crow", csr.crow_indices().tolist())
print("col", csr.col_indices().tolist())
print("vals", csr.values().tolist())

In [ ]:
i = torch.tensor([[0, 1, 1], [0, 0, 2]])
v = torch.tensor([1.0, 2.0, 3.0])
a = torch.sparse_coo_tensor(i, v, (2, 3)).coalesce().to_sparse_csr()
b = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
print(torch.sparse.mm(a, b))
print("ref", a.to_dense() @ b)

In [ ]:
i = torch.tensor([[0, 1], [1, 0]])
v = torch.tensor([0.5, -1.0], requires_grad=True)
w = torch.sparse_coo_tensor(i, v, (2, 2)).coalesce()
x = torch.tensor([[1.0], [2.0]])
loss = (w.to_dense() @ x).square().sum()
loss.backward()
print("grad values", v.grad.tolist())

In [ ]:
def sparsity_report(t):
    nnz = t._nnz() if t.is_sparse else int((t != 0).sum())
    return nnz, nnz / t.numel()

m = torch.zeros(200, 200)
m[torch.randint(0, 200, (80,)), torch.randint(0, 200, (80,))] = 1
print("dense nnz/density", sparsity_report(m))
print("coo nnz/density", sparsity_report(m.to_sparse()))

## Next Steps

- Run `sparse_basics.py` for the full script
- Read the [torch.sparse docs](https://pytorch.org/docs/stable/sparse.html)
- For 2:4 structured sparsity, see Module 31 (torchao)